# Результаты фильтра в Colab

Блокнот строит отчёт по сохранённым файлам прогона. Подготовьте журнал `.jsonl` и одноимённую сводку `.json` доработанной версии. В каждом окне выбирайте два файла сразу.

Для сравнения включите флажок `COMPARE_WITH_BASELINE` и загрузите также пару исходного прогона. Исходный и новый файлы могут называться одинаково.

Версию фильтра указывают настройки или запись о запуске. В первом журнале Colab номер коммита отсутствует.

In [ ]:
# @title Подготовка
from hashlib import sha256
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
from tempfile import mkdtemp
from urllib.request import urlopen
import json
import sys

from google.colab import files
from IPython.display import Markdown, display

ANALYZER_REVISION = "32e49b45f62172e4c6e168e2c675f4b377487195"
ANALYZER_SHA256 = "09a129559f9367330001bed332186d2a2271778fa17c2b57de45296fd1febec5"
ANALYZER_URL = (
    "https://raw.githubusercontent.com/bissonette/analysis-of-injections-using-retrieved-context/"
    f"{ANALYZER_REVISION}/scripts/compare_colab_results.py"
)

cache = Path(mkdtemp(prefix="ctxauth-results-"))
with urlopen(ANALYZER_URL, timeout=30) as response:
    source = response.read()
if sha256(source).hexdigest() != ANALYZER_SHA256:
    raise RuntimeError("Не удалось проверить версию анализатора.")
analyzer_path = cache / "compare_colab_results.py"
analyzer_path.write_bytes(source)
spec = spec_from_file_location("ctxauth_colab_results", analyzer_path)
analyzer = module_from_spec(spec)
sys.modules[spec.name] = analyzer
spec.loader.exec_module(analyzer)


def upload_pair(message, directory):
    print(message)
    selected = [Path(name) for name in files.upload(target_dir=str(directory))]
    journals = [p for p in selected if p.suffix == ".jsonl"]
    if len(selected) != 2 or len(journals) != 1 or journals[0].with_suffix(".json") not in selected:
        raise ValueError("Выберите два файла с одинаковым именем и расширениями .jsonl и .json.")
    return analyzer.load_run(journals[0])


def rate(count, total):
    if not total:
        return "—"
    percent = f"{count / total:.2%}".replace(".", ",")
    return f"{count} из {total} ({percent})"


def summary_rows(report):
    attack = report["attacks"]
    clean = report["clean"]
    task = report["by_permission"].get("task", {"after": 0, "n": 0})
    return {
        "Успешные инъекции без фильтра": rate(attack["before"], attack["n"]),
        "Успешные инъекции с фильтром": rate(attack["after"], attack["n"]),
        "Успешные подмены задачи с фильтром": rate(task["after"], task["n"]),
        "Затронутые чистые примеры": rate(clean["touched"], clean["n"]),
        "Дословно сохранённые чистые ответы": rate(clean["verbatim_preserved"], clean["n"]),
    }

In [ ]:
# @title Загрузка файлов и отчёт
COMPARE_WITH_BASELINE = False  # @param {type:"boolean"}

report_path = None
directory = Path(mkdtemp(prefix="view-", dir=cache))
updated = upload_pair("Загрузите файлы доработанной версии.", directory / "updated")
updated_summary = analyzer.summarize(updated)
updated_rows = summary_rows(updated_summary)
report = {"candidate": updated_summary}

if COMPARE_WITH_BASELINE:
    baseline = upload_pair("Загрузите файлы исходного прогона.", directory / "baseline")
    comparison = analyzer.compare(baseline, updated)
    baseline_summary = analyzer.summarize(baseline)
    report.update({"previous": baseline_summary, "comparison": comparison})
    baseline_rows = summary_rows(baseline_summary)
    table = [
        "| Показатель | Исходный прогон | Доработанный фильтр |",
        "| --- | ---: | ---: |",
        *(f"| {name} | {baseline_rows[name]} | {value} |" for name, value in updated_rows.items()),
    ]
    display(Markdown("\n".join(table)))
    changes = comparison["attacks_with_filter"]
    print(f"Между версиями: {changes['removed']} случаев перестали быть успешными, "
          f"{changes['added']} стали успешными.")
    control = comparison["attacks_without_filter"]
    control_changes = control["removed"] + control["added"]
    if control_changes:
        print(f"В прогонах без фильтра исход различался для {control_changes} примеров. "
              "Учитывайте это при интерпретации разницы.")
else:
    table = [
        "| Показатель | Доработанный фильтр |",
        "| --- | ---: |",
        *(f"| {name} | {value} |" for name, value in updated_rows.items()),
    ]
    display(Markdown("\n".join(table)))
    if updated.unresolved_errors:
        print(f"В журнале {updated.unresolved_errors} незавершённых примеров. "
              "Таблица учитывает только записанные исходы.")

report_path = directory / "colab-results-report.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Отчёт готов к скачиванию.")

In [ ]:
# @title Скачать отчёт
# Отчёт содержит таблицу и разбивки по источникам.
if globals().get("report_path") is None:
    raise ValueError("Сначала сформируйте отчёт в предыдущей ячейке.")
files.download(str(report_path))